# Real-World MCP Servers

*Notebook 28*

Connect to the filesystem and web fetch MCP servers, then combine multiple servers in one agent.
<br>
<br>

**Topics:**
- Filesystem MCP server — read, write, and navigate files

- Web fetch MCP server — retrieve and summarize web content

- Combining multiple MCP servers in one agent

- Reasoning about server permissions — when not to expose a whole filesystem

- Common MCP failure modes — debugging tool discovery and flaky servers

- Testing MCP tools before trusting them in an agent

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner
from agents.mcp import MCPServerStdio, create_static_tool_filter

# Notebook-specific imports
import subprocess

MODEL = "gpt-5-mini"


print("✅ Ready!")

### Check Prerequisites

In [ ]:
checks = {}

# Check Node.js / npx
try:
    node_v = subprocess.check_output(["node", "--version"], text=True).strip()
    checks["Node.js"] = (True, node_v)
except FileNotFoundError:
    checks["Node.js"] = (False, "not found — install from https://nodejs.org")

# Check uvx (for web fetch MCP server)
# uvx is a Python package runner from the 'uv' toolchain — the Python equivalent of npx.
# The web fetch MCP server ships as a Python package and uses it as its runtime.
try:
    uvx_v = subprocess.check_output(["uvx", "--version"], text=True).strip()
    checks["uvx"] = (True, uvx_v)
except FileNotFoundError:
    checks["uvx"] = (False, "not found — install with: pip install uv")

print("Prerequisites:")
for name, (ok, msg) in checks.items():
    icon = "✅" if ok else "❌"
    print(f"  {icon} {name}: {msg}")

all_ok = all(ok for ok, _ in checks.values())
if all_ok:
    print("\n✅ All prerequisites met!")
else:
    print("\n⚠️  Fix the issues above before proceeding.")
    print("   For uvx: run 'pip install uv' in your terminal, then restart JupyterLab")

---

## 📁 Part 1: Filesystem MCP Server

Let's start with the filesystem server in a multi-step read-and-write workflow.

In [ ]:
# Create a workspace with richer content
workspace = Path("mcp_workspace_b")
workspace.mkdir(exist_ok=True)

(workspace / "product_specs.txt").write_text("""Product: TechGadget Pro
Version: 2.0
Release: Q1 2025

Features:
- Bluetooth 5.3 connectivity
- 12-hour battery life
- AI-powered noise cancellation
- Compatible with iOS, Android, Windows, macOS

Pricing:
- Standard: $149.99
- Pro bundle (with case): $179.99
""")

(workspace / "customer_feedback.txt").write_text("""Customer Feedback — January 2025

1. "Battery life is excellent but setup was confusing" — 4/5 stars
2. "Noise cancellation is the best I've used" — 5/5 stars
3. "Would love an Android app" — 3/5 stars
4. "Firmware update bricked my device, support helped fix it" — 3/5 stars
5. "Premium sound quality, worth every penny" — 5/5 stars
""")

# --------------------------------------------------------------
print(f"✅ Workspace created: {workspace.resolve()}")

#### Filesystem Agent Demo

In [ ]:
print("📁 FILESYSTEM MCP DEMO")
print("=" * 60)

async with MCPServerStdio(
    name="Filesystem",
    params={
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", str(workspace.resolve())]
    },
    cache_tools_list=True
) as fs_server:

    product_analyst_instructions = (
        "You are a product analyst. Read the files in the workspace and produce\n"
        "a concise analysis report covering product specs and customer sentiment."
    )

    agent = Agent(
        name="ProductAnalyst",
        instructions=product_analyst_instructions,
        model=MODEL,
        mcp_servers=[fs_server]
    )

    result = await Runner.run(agent, input="Read the product specs and customer feedback files and write a brief analysis report to analysis_report.txt")

    print(result.final_output)

# Verify the report was written
report = workspace / "analysis_report.txt"
if report.exists():
    print(f"\n✅ Report written ({len(report.read_text())} chars)")

print("=" * 60)

⚠️ **Security note:** The filesystem MCP server has write access.

Passing `/` or `~` would give the agent access to your entire system — credentials, configs, private files.

Scope it to a dedicated workspace directory to limit the blast radius. (See Lesson 23 for prompt injection risks with tool output.)

---

## 🌐 Part 2: Web Fetch MCP Server

The web fetch MCP server gives the agent a `fetch` tool that retrieves any URL and returns the content as clean markdown — ready for the agent to read and reason about.

In [ ]:
print("🌐 WEB FETCH MCP DEMO")
print("=" * 60)

async with MCPServerStdio(
    name="WebFetch",
    params={
        "command": "uvx",
        "args": ["mcp-server-fetch"]
    },
    cache_tools_list=True
) as fetch_server:

    # Show what tools the fetch server exposes
    tools = await fetch_server.list_tools()
    print(f"Fetch server tools: {[t.name for t in tools]}\n")

    web_researcher_instructions = (
        "You are a web researcher. Use the fetch tool to retrieve web pages\n"
        "and provide concise summaries of their content."
    )

    agent = Agent(
        name="WebResearcher",
        instructions=web_researcher_instructions,
        model=MODEL,
        mcp_servers=[fetch_server]
    )

    result = await Runner.run(agent, input="Fetch https://openai.com and give me a 3-sentence summary of what OpenAI does.")

    print(result.final_output)
    print("=" * 60)

⚠️ **Security note:** Content returned by the fetch tool comes from the web — treat it as untrusted input.

A malicious page could contain text designed to override the agent's instructions.

Apply the same prompt injection awareness from Lesson 23 — keep system instructions separate from fetched content, and validate or constrain what the agent does with retrieved data.

---

## 🔀 Part 3: Combining Multiple MCP Servers

Pass multiple servers to `mcp_servers=[]`.

The agent aggregates all tools from all servers and uses them together.

This is the payoff of the MCP standard — you can give an agent local file access and web access by composing two servers someone else wrote, with zero integration code on your side.

In [ ]:
print("🔀 COMBINED MCP SERVERS DEMO")
print("=" * 60)

async with (
    MCPServerStdio(
        name="Filesystem",
        params={
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", str(workspace.resolve())]
        },
        cache_tools_list=True
    ) as fs_server,
    MCPServerStdio(
        name="WebFetch",
        params={
            "command": "uvx",
            "args": ["mcp-server-fetch"]
        },
        cache_tools_list=True
    ) as fetch_server
):
    research_assistant_instructions = (
        "You are a research assistant with access to local files and the web.\n"
        "Use filesystem tools to read and write files.\n"
        "Use the fetch tool to retrieve web content.\n"
        "Combine both as needed to complete research tasks."
    )

    agent = Agent(
        name="ResearchAssistant",
        instructions=research_assistant_instructions,
        model=MODEL,
        mcp_servers=[fs_server, fetch_server]
    )

    result = await Runner.run(agent, input="Read the product_specs.txt file from the workspace. Then fetch https://www.python.org and find the current Python version. Finally, save a combined_research.txt file with both findings.")

    print(result.final_output)

# Verify combined file was written
combined = workspace / "combined_research.txt"
if combined.exists():
    print(f"\n✅ combined_research.txt created ({len(combined.read_text())} chars)")

print("=" * 60)

---

## 🛠️ Part 4: Tool Filtering

MCP servers can expose many tools.

Sometimes you only want to give the agent access to a subset — for safety, focus, or simplicity.

Use `create_static_tool_filter` to allowlist specific tools.

`create_static_tool_filter(allowed_tool_names=[...])` tells the SDK to hide any tools not on your list — the agent literally cannot see or call them.

Scoping the directory limits *which files* the agent can reach — filtering tools limits *what the agent can do* with those files (two complementary safety layers).

This is safer than instructions alone: even if a later prompt says "delete everything," the write tools are invisible to the agent.

In [ ]:
print("🛠️ TOOL FILTERING DEMO")
print("=" * 60)

async with MCPServerStdio(
    name="ReadOnlyFilesystem",
    params={
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", str(workspace.resolve())]
    },
    cache_tools_list=True,
    # Only allow read and list tools — no write or delete
    tool_filter=create_static_tool_filter(
        allowed_tool_names=["read_file", "list_directory", "get_file_info"]
    )
) as readonly_fs:

    tools = await readonly_fs.list_tools()
    # In your own project, call list_tools() on the unfiltered server first to see
    # the exact tool names, then put only the ones you want into allowed_tool_names.
    print(f"Filtered tool list ({len(tools)} tools):")
    for tool in tools:
        print(f"  ✅ {tool.name}")

    agent = Agent(
        name="ReadOnlyAgent",
        instructions="You can only read files — no writing allowed.",
        model=MODEL,
        mcp_servers=[readonly_fs]
    )

    result = await Runner.run(agent, input="List the files and read customer_feedback.txt")

    print(f"\nAgent response: {result.final_output}")

print("=" * 60)

---

## 🔍 Part 5: Testing MCP Servers Before Trusting Them

Before wiring an MCP server into a production agent, verify it behaves as expected.

**Checklist:**
- Start the server and confirm it connects cleanly — watch for startup errors
- Call `list_tools()` and verify the expected tools appear
- Test a tool directly (outside an agent) before putting it in `mcp_servers=[]`. In practice: connect to the server, call `list_tools()` to confirm what it exposes, optionally invoke one tool to see its raw output, then wire it into an `Agent`.
- Treat all tool-returned content as untrusted input — web pages, file contents, and database results can all contain text designed to manipulate the agent (prompt injection)
- Restrict permissions to the minimum needed — read-only unless write is required
- If a tool behaves unexpectedly, check the server logs before assuming a model issue

**Common failure modes:**
- **Server won't start** — wrong `command` or missing runtime (Node.js/uvx not installed); check with `node --version` or `uvx --version` before running
- **Tool not in list** — server started but the expected tool name is absent; run `list_tools()` directly to see what the server actually exposes
- **Timeout on first call** — `npx -y` downloads the package on first run (15–30s); subsequent runs are fast
- **Unexpected tool behavior** — test tools directly outside an agent first; isolates whether the issue is the tool or the agent's reasoning

---

## 🧹 Cleanup

In [ ]:
import shutil
shutil.rmtree(workspace, ignore_errors=True)
print("✅ Workspace cleaned up")

---

## 💪 Practice Exercises

### Exercise 1: Research and Save

*Covers: MCP fetch server — web retrieval and file saving*

Build an agent that fetches a URL and saves a summary to a local file.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Research and Save
# --------------------------------------------------------------
# Objective: Fetch a web page and save a summary to a local file.

# TODO 1: Create a workspace directory

# TODO 2: Open both MCPServerStdio (filesystem) and MCPServerStdio (fetch)
#           in a combined async with block

# TODO 3: Create an agent with both servers

# TODO 4: Ask the agent to:
#           - Fetch https://docs.python.org/3/
#           - Summarize what Python documentation covers
#           - Save the summary to python_docs_summary.txt

# TODO 5: Print the saved file contents and clean up

# --- Write your code below this line ---

### Exercise 2: Read-Only Research Agent

*Covers: MCP security — enforcing read-only access*

Build a read-only agent that can only read files from a local directory — no writing allowed.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Read-Only Research Agent
# --------------------------------------------------------------
# Objective: Use tool filtering to restrict the agent to read-only access.

# TODO 1: Create a workspace with 2-3 text files of your choosing

# TODO 2: Connect to the filesystem MCP server with create_static_tool_filter
#           allowed_tool_names: only include read_file and list_directory

# TODO 3: Verify the tool list is filtered (print the allowed tools)

# TODO 4: Create an agent and ask it to summarize all files

# TODO 5: (Optional) Try asking it to write a file — observe what happens

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Filesystem MCP Server:**

- `npx -y @modelcontextprotocol/server-filesystem <path>` — scoped to one directory

- Exposes read, write, list, and search tools automatically
<br>
<br>

**Web Fetch MCP Server:**

- `uvx mcp-server-fetch` — retrieves URLs and returns clean markdown

- No API key needed — just install `uv` and run
<br>
<br>

**Combining Servers:**

- `mcp_servers=[server1, server2]` — agent sees all tools from all servers

- Use Python's `async with` multiple context managers for clean lifecycle management
<br>
<br>

**Tool Filtering:**

- `create_static_tool_filter(allowed_tool_names=[...])` — expose only what you need

- Filtered tools are invisible to the agent — it can't use what it can't see
<br>
<br>

**Safety and Trust:**

- Treat all tool-returned content as untrusted input — web pages and file contents can contain text designed to manipulate the agent (prompt injection)

- Test MCP servers with `list_tools()` and direct tool calls before wiring them into a production agent

---

## 📍 Next Step

**Notebook 29: Capstone #4 — MCP-Powered Personal Assistant**  

Connect to filesystem, web, and a third MCP server to build a real-world assistant that can research, read, and save results.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-28-real-world-mcp-servers)

---